# Random Forest Baseline — Stroke Prevention Demo

DP-12

## Purpose
Train a Random Forest classifier on the same cleaned dataset and split as the logistic regression baseline (DP-11).
Report the same four metrics so results are directly comparable.

## How to run
Run cells top to bottom. Requires  in the repo root.

## What Random Forest is
A Random Forest trains many decision trees on random subsets of the data and features, then averages their predictions.
It handles non-linear relationships and feature interactions automatically, without needing manual feature engineering.
StandardScaler has no effect on tree splits but we keep the same preprocessing pipeline for a fair comparison.

## 1. Config

Settings that mirror the baseline are marked `# same as baseline`.
Do not change `TEST_SIZE`, `RANDOM_STATE`, or column lists unless the baseline changes.

In [1]:
from pathlib import Path
import numpy as np

# --- Split settings (same as baseline) ---
TEST_SIZE    = 0.20  # same as baseline
RANDOM_STATE = 42    # same as baseline
TRY_STRATIFY = True  # same as baseline

# Threshold for converting probability to 0/1 prediction (same as baseline)
THRESHOLD = 0.30

# Paths
DATA_REL_PATH   = Path("data/raw/stroke_data.csv")
REPORT_REL_PATH = Path("reports/random_forest_comparison.md")

# --- Leakage / exclusion columns (same as baseline) ---
LEAKAGE_COLS = [
    "General health condition",
    "depression",
    "Minutes sedentary activity",
    "Coronary Heart Disease",
]

EXCLUDE_COLS = [
    "High-density lipoprotein",
    "Triglyceride",
    "Low-density lipoprotein",
]

REDUNDANT_COLS = ["Total fat"]

# --- Categorical column assignments (same as baseline) ---
NOMINAL_CATEGORICAL_COLS = [
    "gender",
    "Race",
    "Marital status",
    "sleep disorder",
    "Health Insurance",
    "Body Mass Index",
]

ORDINAL_COLS       = ["age"]
ORDINAL_CATEGORIES = [[1, 2, 3]]

# --- Dietary zeros treated as missing (same as baseline) ---
ZERO_AS_MISSING_COLS = [
    "energy", "protein", "Carbohydrate", "Dietary fiber",
    "Total saturated fatty acids", "Total monounsaturated fatty acids",
    "Total polyunsaturated fatty acids", "Potassium", "Sodium",
]

# --- Physiological hard caps (same as baseline) ---
PHYSIOLOGICAL_CAPS = {
    "energy":                             (400,   6000),
    "protein":                            (5,     280),
    "Carbohydrate":                       (5,     700),
    "Dietary fiber":                      (1,     70),
    "Total saturated fatty acids":        (0.5,   100),
    "Total monounsaturated fatty acids":  (0.5,   100),
    "Total polyunsaturated fatty acids":  (0.5,   75),
    "Potassium":                          (200,   8000),
    "Sodium":                             (300,   12000),
    "Glycohemoglobin":                    (3.5,   18.0),
}

# --- IQR outlier clipping (same as baseline) ---
CLIP_OUTLIERS       = True
CLIP_IQR_MULTIPLIER = 2.0

# --- Random Forest settings ---
# Simple first attempt: 100 trees, balanced weights, no hyperparameter search.
RF_N_ESTIMATORS   = 100
RF_CLASS_WEIGHT   = "balanced"
RF_MAX_DEPTH      = None  # grow fully, controlled by min_samples_leaf
RF_MIN_SAMPLES_LEAF = 5   # small guard against a single noisy leaf

# Baseline numbers from reports/baseline_metrics.md (used in comparison table)
BASELINE = {
    "model":     "Logistic Regression (elastic net, C=0.1)",
    "accuracy":  0.3550,
    "precision": 0.0960,
    "recall":    0.8611,
    "roc_auc":   0.5830,
}

## 2. Imports

In [2]:
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, StandardScaler

## 3. Load data

In [3]:
def find_repo_root() -> Path:
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / DATA_REL_PATH).exists():
            return p
    return here

ROOT      = find_repo_root()
DATA_PATH = ROOT / DATA_REL_PATH

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Could not find dataset at: {DATA_PATH}")

df = pd.read_csv(DATA_PATH)
print(f"Loaded {len(df):,} rows, {df.shape[1]} columns")
df.head()

Loaded 4,603 rows, 36 columns


,stroke,gender,age,Race,Marital status,alcohol,smoke,sleep disorder,Health Insurance,General health condition,...,energy,protein,Carbohydrate,Dietary fiber,Total fat,Total saturated fatty acids,Total monounsaturated fatty acids,Total polyunsaturated fatty acids,Potassium,Sodium
0,0,2,2,5,1,0,0,2,2,3,...,1598,62.78,192.19,10.0,65.64,25.112,24.090,8.543,2887,2969
1,0,2,2,1,1,0,0,1,2,3,...,1547,45.35,256.02,17.0,42.56,13.423,15.389,10.613,2058,2091
2,1,1,2,3,1,1,1,2,1,3,...,2466,81.56,254.49,13.0,103.32,43.295,36.727,15.366,3117,5233
3,0,2,3,3,1,1,1,2,1,4,...,1605,70.99,143.37,10.0,81.60,24.527,30.567,18.174,1766,3706
4,0,1,1,4,1,0,0,2,1,2,...,1818,74.75,229.45,14.2,67.49,26.030,24.837,10.533,1842,2461


## 3b. Data cleaning

Exact same steps as the baseline notebook:
1. Strip column name whitespace
2. Drop duplicate rows
3. Replace zero with NaN in dietary columns (zero daily intake is physiologically impossible)
4. Apply physiological hard caps

In [4]:
# Strip whitespace from column names
df.columns = df.columns.str.strip()

# Drop duplicate rows
n_dupes = df.duplicated().sum()
if n_dupes > 0:
    df = df.drop_duplicates().reset_index(drop=True)
    print(f"Dropped {n_dupes} duplicate rows.")

# Replace 0 with NaN in dietary columns
zero_cols_present = [c for c in ZERO_AS_MISSING_COLS if c in df.columns]
df[zero_cols_present] = df[zero_cols_present].replace(0, np.nan)

# Apply physiological hard caps
for col, (lo, hi) in PHYSIOLOGICAL_CAPS.items():
    if col in df.columns:
        df[col] = df[col].clip(lower=lo, upper=hi)

print(f"Cleaning done. Dataset shape: {df.shape}")

Cleaning done. Dataset shape: (4603, 36)


## 3c. Feature engineering

Same engineered features as the baseline:
- **multimorbidity**: count of concurrent risk conditions (diabetes + hypertension + high cholesterol + smoke)
- **age interaction terms**: age x hypertension, age x smoke, age x high cholesterol

In [5]:
_risk_flags = ["diabetes", "hypertension", "high cholesterol", "smoke"]
_present    = [c for c in _risk_flags if c in df.columns]
df["multimorbidity"]         = df[_present].sum(axis=1)
df["age_x_hypertension"]     = df["age"] * df["hypertension"]
df["age_x_smoke"]            = df["age"] * df["smoke"]
df["age_x_high_cholesterol"] = df["age"] * df["high cholesterol"]

print("Engineered: multimorbidity, age_x_hypertension, age_x_smoke, age_x_high_cholesterol")
print(f"Dataset shape after feature engineering: {df.shape}")

Engineered: multimorbidity, age_x_hypertension, age_x_smoke, age_x_high_cholesterol
Dataset shape after feature engineering: (4603, 40)


## 4. Define label and features

In [6]:
drop_cols         = [c for c in LEAKAGE_COLS   if c in df.columns]
exclude_present   = [c for c in EXCLUDE_COLS   if c in df.columns]
redundant_present = [c for c in REDUNDANT_COLS if c in df.columns]

df_clean = df.drop(columns=drop_cols + exclude_present + redundant_present)

y = df_clean["stroke"]
X = df_clean.drop(columns=["stroke"])

print(f"Features: {X.shape[1]}")
print()
print("Label counts")
print(y.value_counts(dropna=False))
print()
print("Label proportions")
print(y.value_counts(normalize=True, dropna=False).round(4))

Features: 31

Label counts
stroke
0    4241
1     362
Name: count, dtype: int64

Label proportions
stroke
0    0.9214
1    0.0786
Name: proportion, dtype: float64


## 5. Train/test split

Identical settings to baseline: `test_size=0.20`, `random_state=42`, `stratify=yes`

In [7]:
try:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        stratify=y if TRY_STRATIFY else None,
    )
    stratify_used = "yes"
except ValueError as e:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE,
    )
    stratify_used = "no"
    print(f"Stratify fallback: {e}")

print(f"Split: test_size={TEST_SIZE}, seed={RANDOM_STATE}, stratify={stratify_used}")
print(f"Train: {len(X_train):,} rows | Test: {len(X_test):,} rows")

Split: test_size=0.2, seed=42, stratify=yes
Train: 3,682 rows | Test: 921 rows


## 5b. Outlier clipping

Same IQR-based clipping as baseline (k=2.0). Bounds computed on training set only to prevent leakage.

In [8]:
if CLIP_OUTLIERS:
    all_cat_cols  = ORDINAL_COLS + NOMINAL_CATEGORICAL_COLS
    clip_num_cols = [c for c in X_train.columns if c not in all_cat_cols]

    Q1  = X_train[clip_num_cols].quantile(0.25)
    Q3  = X_train[clip_num_cols].quantile(0.75)
    IQR = Q3 - Q1
    lower_bounds = Q1 - CLIP_IQR_MULTIPLIER * IQR
    upper_bounds = Q3 + CLIP_IQR_MULTIPLIER * IQR

    X_train = X_train.copy()
    X_test  = X_test.copy()
    X_train[clip_num_cols] = X_train[clip_num_cols].clip(lower=lower_bounds, upper=upper_bounds, axis=1)
    X_test[clip_num_cols]  = X_test[clip_num_cols].clip(lower=lower_bounds, upper=upper_bounds, axis=1)

    print(f"Outlier clipping applied (k={CLIP_IQR_MULTIPLIER})")

Outlier clipping applied (k=2.0)


## 6. Preprocessing pipeline

Same three-stream ColumnTransformer as baseline:
- **Ordinal** (): most-frequent impute -> OrdinalEncoder
- **Nominal** (gender, Race, etc.): most-frequent impute -> OneHotEncoder
- **Numeric** (all others): median impute -> StandardScaler

Note: StandardScaler does not affect Random Forest tree splits, but keeping it
makes the preprocessing pipeline identical to the baseline for a fair comparison.

In [9]:
ord_cols = [c for c in ORDINAL_COLS             if c in X.columns]
nom_cols = [c for c in NOMINAL_CATEGORICAL_COLS if c in X.columns]
num_cols = [c for c in X.columns if c not in ord_cols + nom_cols]

print(f"Ordinal:  {ord_cols}")
print(f"Nominal:  {nom_cols}")
print(f"Numeric:  {len(num_cols)} columns")

ord_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ordinal", OrdinalEncoder(categories=ORDINAL_CATEGORIES)),
])

nom_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot",  OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

num_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  StandardScaler()),
])

preprocess = ColumnTransformer(transformers=[
    ("ord", ord_pipe, ord_cols),
    ("nom", nom_pipe, nom_cols),
    ("num", num_pipe, num_cols),
])

Ordinal:  ['age']
Nominal:  ['gender', 'Race', 'Marital status', 'sleep disorder', 'Health Insurance', 'Body Mass Index']
Numeric:  24 columns


## 7. Random Forest model

Settings:
-  — 100 trees, a standard starting point
-  — same as baseline; upweights the rare stroke class (~7.9% of data)
-  — small guard against a single leaf memorising noise
-  — trees grow fully, controlled by 
- No grid search — this is a first honest attempt, not an optimised model

In [10]:
rf_model = RandomForestClassifier(
    n_estimators=RF_N_ESTIMATORS,
    class_weight=RF_CLASS_WEIGHT,
    max_depth=RF_MAX_DEPTH,
    min_samples_leaf=RF_MIN_SAMPLES_LEAF,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

pipeline = Pipeline([
    ("preprocess", preprocess),
    ("model",      rf_model),
])

pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocess', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('ord', ...), ('nom', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers

## 8. Train

In [11]:
pipeline.fit(X_train, y_train)
print("Training complete.")

Training complete.


## 9. Evaluate on test set

Same threshold (0.30) and same four metrics as baseline.

Metrics:
- **Accuracy**: percent of all predictions correct
- **Precision**: when we flag stroke, how often we are right
- **Recall**: out of true stroke cases, how many we caught
- **ROC AUC**: how well the probability scores rank stroke vs non-stroke (threshold-independent)

In [12]:
y_prob = pipeline.predict_proba(X_test)[:, 1]
y_pred = (y_prob >= THRESHOLD).astype(int)

rf_accuracy  = float(accuracy_score(y_test, y_pred))
rf_precision = float(precision_score(y_test, y_pred, zero_division=0))
rf_recall    = float(recall_score(y_test, y_pred, zero_division=0))
rf_auc       = float(roc_auc_score(y_test, y_prob))
rf_cm        = confusion_matrix(y_test, y_pred)

print(f"Threshold  : {THRESHOLD}")
print(f"Accuracy   : {rf_accuracy:.4f}")
print(f"Precision  : {rf_precision:.4f}")
print(f"Recall     : {rf_recall:.4f}")
print(f"ROC AUC    : {rf_auc:.4f}")
print()
print("Confusion matrix [[TN FP] / [FN TP]]")
print(rf_cm)

Threshold  : 0.3
Accuracy   : 0.7872
Precision  : 0.0921
Recall     : 0.1944
ROC AUC    : 0.5692

Confusion matrix [[TN FP] / [FN TP]]
[[711 138]
 [ 58  14]]


## 10. Feature importances

Random Forest provides impurity-based feature importances (mean decrease in Gini impurity across all trees).
These show which features the model relied on most.

In [13]:
feature_names = pipeline.named_steps["preprocess"].get_feature_names_out()
importances   = pipeline.named_steps["model"].feature_importances_

importance_df = (
    pd.DataFrame({"feature": feature_names, "importance": importances})
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

print("Top 15 features by importance:")
print(importance_df.head(15).to_string(index=False))

Top 15 features by importance:
                               feature  importance
                    num__Dietary fiber    0.056904
num__Total polyunsaturated fatty acids    0.055966
      num__Total saturated fatty acids    0.054955
num__Total monounsaturated fatty acids    0.051827
                          num__protein    0.051239
                     num__Carbohydrate    0.051050
          num__Systolic blood pressure    0.049841
              num__Waist Circumference    0.048839
                           num__Sodium    0.048541
                  num__Fasting Glucose    0.048351
                        num__Potassium    0.048227
         num__Diastolic blood pressure    0.046801
                           num__energy    0.045551
                              ord__age    0.039244
                  num__Glycohemoglobin    0.037280


## 11. Comparison table

Side-by-side with the logistic regression baseline (from `reports/baseline_metrics.md`).
Same threshold (0.30), same split, same features.

In [15]:
comparison = pd.DataFrame([
    {
        "Model":     BASELINE["model"],
        "Accuracy":  BASELINE["accuracy"],
        "Precision": BASELINE["precision"],
        "Recall":    BASELINE["recall"],
        "ROC AUC":   BASELINE["roc_auc"],
    },
    {
        "Model":     f"Random Forest (n={RF_N_ESTIMATORS}, balanced, min_leaf={RF_MIN_SAMPLES_LEAF})",
        "Accuracy":  rf_accuracy,
        "Precision": rf_precision,
        "Recall":    rf_recall,
        "ROC AUC":   rf_auc,
    },
])

print(comparison.to_string(index=False, float_format="{:.4f}".format))

                                      Model  Accuracy  Precision  Recall  ROC AUC
   Logistic Regression (elastic net, C=0.1)    0.3550     0.0960  0.8611   0.5830
Random Forest (n=100, balanced, min_leaf=5)    0.7872     0.0921  0.1944   0.5692


## 12. Save comparison report

In [16]:
REPORT_PATH = ROOT / REPORT_REL_PATH
REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)

delta_auc       = rf_auc       - BASELINE["roc_auc"]
delta_recall    = rf_recall    - BASELINE["recall"]
delta_precision = rf_precision - BASELINE["precision"]
delta_accuracy  = rf_accuracy  - BASELINE["accuracy"]

def fmt_delta(d):
    return f"+{d:.4f}" if d >= 0 else f"{d:.4f}"

if delta_auc > 0.01:
    verdict = "Random Forest shows a meaningful improvement in ROC AUC over the logistic regression baseline."
elif delta_auc < -0.01:
    verdict = "Logistic Regression outperforms Random Forest on ROC AUC; the baseline remains stronger."
else:
    verdict = "Random Forest and logistic regression perform similarly on ROC AUC (within 0.01)."

if rf_recall > BASELINE["recall"]:
    recall_verdict = "Random Forest catches more true stroke cases (higher recall)."
elif rf_recall < BASELINE["recall"]:
    recall_verdict = "Logistic Regression catches more true stroke cases (higher recall)."
else:
    recall_verdict = "Both models have identical recall."

demo_recommendation = (
    "Random Forest is recommended for the demo if ROC AUC and recall are both higher."
    if (rf_auc > BASELINE["roc_auc"] and rf_recall >= BASELINE["recall"])
    else (
        "Logistic Regression remains the stronger demo model."
        if rf_auc <= BASELINE["roc_auc"]
        else "Consider a tradeoff: Random Forest ranks better (AUC) but catches fewer true positives (recall)."
    )
)

with open(REPORT_PATH, "w", encoding="utf-8") as f:
    f.write("# Random Forest vs Baseline Comparison")

    f.write("## Split settings (identical for both models)")
    f.write(f"- test_size: {TEST_SIZE}")
    f.write(f"- random_state: {RANDOM_STATE}")
    f.write(f"- stratify: {stratify_used}")
    f.write(f"- threshold: {THRESHOLD}")

    f.write("## Metrics at threshold 0.30")
    f.write("| Metric    | Baseline (LR) | Random Forest | Delta |")
    f.write("|-----------|:-------------:|:-------------:|:-----:|")
    f.write(f"| Accuracy  | {BASELINE['accuracy']:.4f}        | {rf_accuracy:.4f}         | {fmt_delta(delta_accuracy)} |")
    f.write(f"| Precision | {BASELINE['precision']:.4f}        | {rf_precision:.4f}         | {fmt_delta(delta_precision)} |")
    f.write(f"| Recall    | {BASELINE['recall']:.4f}        | {rf_recall:.4f}         | {fmt_delta(delta_recall)} |")
    f.write(f"| ROC AUC   | {BASELINE['roc_auc']:.4f}        | {rf_auc:.4f}         | {fmt_delta(delta_auc)} |")

    f.write("## Random Forest confusion matrix")
    f.write("Format: [[TN FP] / [FN TP]]")
    f.write(f"{rf_cm.tolist()}")

    f.write("## Random Forest setting")
    f.write(f"- n_estimators: {RF_N_ESTIMATORS}")
    f.write(f"- class_weight: {RF_CLASS_WEIGHT}")
    f.write(f"- max_depth: {RF_MAX_DEPTH}")
    f.write(f"- min_samples_leaf: {RF_MIN_SAMPLES_LEAF}")

    f.write("## Top 10 features by importance")
    for _, row in importance_df.head(10).iterrows():
        f.write(f"- {row['feature']}: {row['importance']:.4f}")
    f.write("")

    f.write("## Summary")
    f.write(
        f"Random Forest (100 trees, balanced weights, min_samples_leaf=5) scored ROC AUC {rf_auc:.4f} "
        f"vs the logistic regression baseline of {BASELINE['roc_auc']:.4f} (delta {fmt_delta(delta_auc)}). "
        f"Recall changed from {BASELINE['recall']:.4f} to {rf_recall:.4f} and precision from "
        f"{BASELINE['precision']:.4f} to {rf_precision:.4f}. "
        f"{verdict} {recall_verdict} "
        f"{demo_recommendation}"
    )

print(f"Report saved to {REPORT_PATH}")

Report saved to /Users/naveedahmed/Desktop/NAVSProjects/stroke-prevention-demo/reports/random_forest_comparison.md
